# Notebook 2: Query Functions and Procedure Checks

Use this notebook to validate query tasks for the current capstone schema.

Checks:
- Show all tables
- Run a JOIN query for customers with orders above `$60`
- Call `GetMaxQuantity()`


In [1]:
import os
from dotenv import find_dotenv, load_dotenv
import mysql.connector as connector

In [2]:
dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError(
        "No .env file found. Copy .env.example to .env and set database credentials."
    )

load_dotenv(dotenv_path)
print(f"Loaded env from: {dotenv_path}")

Loaded env from: /Users/viacheslav/projects/personal/database-engineer-capstone/.env


In [3]:
required_vars = ["DB_HOST", "DB_PORT", "DB_USER", "DB_PASSWORD", "DB_NAME"]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise ValueError(f"Missing env vars: {', '.join(missing)}")

connection = connector.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "3306")),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
)
cursor = connection.cursor()
print("Connected:", connection.is_connected())

Connected: True


## Step 1: Show all tables


In [4]:
# Ensure the connection is still alive before querying
if not connection.is_connected():
    connection.reconnect(attempts=3, delay=1)

show_tables_query = "SHOW TABLES"
cursor.execute(show_tables_query)
results = cursor.fetchall()
print(results)


[('Bookings',), ('Customers',), ('Orders',)]


## Step 2: JOIN query for customers with orders over $10


In [5]:
# Recreate cursor if needed after reconnect
if not connection.is_connected():
    connection.reconnect(attempts=3, delay=1)
    cursor = connection.cursor()

join_query = """
SELECT
    CONCAT(c.FirstName, ' ', c.LastName) AS FullName,
    c.Phone,
    c.Email,
    o.TotalCost
FROM Customers c
INNER JOIN Orders o ON c.CustomerID = o.CustomerID
WHERE o.TotalCost >= 10
ORDER BY o.TotalCost DESC;
"""

cursor.execute(join_query)
rows = cursor.fetchall()
for row in rows:
    print(row)


('Anna Miller', '+1-202-555-0101', 'anna.miller@example.com', Decimal('86.00'))
('David Nguyen', '+1-202-555-0104', 'david.nguyen@example.com', Decimal('43.00'))
('Carla Lopez', '+1-202-555-0103', 'carla.lopez@example.com', Decimal('40.00'))
('Brian Clark', '+1-202-555-0102', 'brian.clark@example.com', Decimal('37.00'))
('Brian Clark', '+1-202-555-0102', 'brian.clark@example.com', Decimal('37.00'))


## Step 3: Call `GetMaxQuantity()`


In [6]:
if not connection.is_connected():
    connection.reconnect(attempts=3, delay=1)
    cursor = connection.cursor()

cursor.execute("CALL GetMaxQuantity();")
for row in cursor.fetchall():
    print("MaxQuantity:", row[0])


MaxQuantity: 3


## Step 4: Run booking procedures (`AddBooking`, `UpdateBooking`, `ManageBooking`, `CancelBooking`)


In [7]:
from mysql.connector import Error

if not connection.is_connected():
    connection.reconnect(attempts=3, delay=1)
    cursor = connection.cursor()

# Use high BookingIDs to avoid collisions with seed data
test_add_id = 90
test_manage_conflict_id = 91
test_manage_ok_id = 92

procedure_calls = [
    ("AddBooking", f"CALL AddBooking({test_add_id}, 1, '2022-12-30', 7);"),
    ("UpdateBooking", f"CALL UpdateBooking({test_add_id}, '2022-12-31');"),
    ("ManageBooking (conflict expected)", f"CALL ManageBooking({test_manage_conflict_id}, 2, '2022-12-31', 7);"),
    ("ManageBooking (success expected)", f"CALL ManageBooking({test_manage_ok_id}, 2, '2022-12-31', 8);"),
    ("CancelBooking", f"CALL CancelBooking({test_add_id});"),
]

def execute_call(label, sql):
    print(f"\n{label} -> {sql}")
    try:
        cursor.execute(sql)
        # Drain all result sets from CALL statements to avoid command-sync errors.
        rows = cursor.fetchall()
        if rows:
            for row in rows:
                print(row)
        while cursor.nextset():
            extra_rows = cursor.fetchall()
            if extra_rows:
                for row in extra_rows:
                    print(row)
        connection.commit()
    except Error as err:
        connection.rollback()
        # Best effort: clear pending results after an error.
        try:
            while cursor.nextset():
                _ = cursor.fetchall()
        except Exception:
            pass
        print(f"ERROR: {err}")

for label, sql in procedure_calls:
    execute_call(label, sql)

# Optional cleanup for rows created by ManageBooking tests
for cleanup_id in (test_manage_conflict_id, test_manage_ok_id):
    execute_call(f"Cleanup {cleanup_id}", f"CALL CancelBooking({cleanup_id});")



AddBooking -> CALL AddBooking(90, 1, '2022-12-30', 7);
('New booking added',)

UpdateBooking -> CALL UpdateBooking(90, '2022-12-31');
('Booking 90 updated',)

ManageBooking (conflict expected) -> CALL ManageBooking(91, 2, '2022-12-31', 7);
('Table 7 is already booked - booking cancelled',)

ManageBooking (success expected) -> CALL ManageBooking(92, 2, '2022-12-31', 8);
('Booking 92 saved successfully',)

CancelBooking -> CALL CancelBooking(90);
('Booking 90 cancelled',)

Cleanup 91 -> CALL CancelBooking(91);
('Booking 91 not found',)

Cleanup 92 -> CALL CancelBooking(92);
('Booking 92 cancelled',)


## Step 5: Verify booking table state after procedure tests


In [8]:
cursor.execute("SELECT BookingID, BookingDate, TableNumber, CustomerID FROM Bookings ORDER BY BookingID;")
for row in cursor.fetchall():
    print(row)


(1, datetime.date(2022, 10, 10), 5, 1)
(2, datetime.date(2022, 11, 12), 3, 3)
(3, datetime.date(2022, 10, 11), 2, 2)
(4, datetime.date(2022, 10, 13), 2, 1)
(11, datetime.date(2022, 12, 30), 4, 2)


## Cleanup

In [9]:
cursor.close()
connection.close()
print("Connection closed")

Connection closed
